Reference: https://maartengr.github.io/BERTopic/index.html#installation

In [288]:
from bertopic import BERTopic
import ssl
import pandas as pd
from umap import UMAP
from typing import List, Union
from sklearn.preprocessing import MinMaxScaler

import plotly.express as px
import plotly.graph_objects as go

# Disable SSL verification - because it is annoying
ssl._create_default_https_context = ssl._create_unverified_context

from sklearn.datasets import fetch_20newsgroups
docs = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))['data']

In [289]:
#data 
df = pd.read_csv('../output/jury_output_chat_level.csv')
df['message'] = df['message'].astype(str)


#convert everything to strings

#split into 3 DFs
split_size = len(df) //3
print(len(df))
print(split_size)

# Split the DataFrame into three parts
df1 = df[:split_size]
df2 = df[split_size:2 * split_size]
df3 = df[2 * split_size:]

14888
4962


In [290]:
'''
Load the datasets at join them convert the message column into strings. 
'''

df_csop1 = pd.read_csv('../output/csop_output_chat_level.csv')
df_csop2 = pd.read_csv('../output/csopII_output_chat_level.csv')
df_csop = df_csop1.append(df_csop2, ignore_index=True)

df_jury = pd.read_csv('../output/jury_output_chat_level.csv')

df_csop['message'] = df_csop['message'].astype(str)
df_jury['message'] = df_jury['message'].astype(str)


In [291]:
#load and transform data - not currently used in this workbook
def load_and_transform_data(path):
    df = pd.read_csv(path)
    df['message'] = df['message'].astype(str)
    return df

In [292]:
'''
Fit the model based on the message column. Each message is a document 
'''
def fit_model(df):
    documents = df['message'].tolist()
    model = BERTopic()
    topics = model.fit_transform(documents)
    return model

In [293]:
'''
Get the top n topics from the model
'''
def get_top_topics(topic_model,n):
    for i in range(0, n): 
        topic_list = topic_model.get_topic(i)
        print("Topic Number "+str(i))
        for key,value in topic_list:
            print(key)
        print("-------------------")

In [294]:
def visualize_topics(topic_model,
                     topics: List[int] = None,
                     top_n_topics: int = None,
                     custom_labels: Union[bool, str] = False,
                     title: str = "<b>Intertopic Distance Map</b>",
                     width: int = 650,
                     height: int = 650) -> go.Figure:
    """ Visualize topics, their sizes, and their corresponding words

    This visualization is highly inspired by LDAvis, a great visualization
    technique typically reserved for LDA.

    Arguments:
        topic_model: A fitted BERTopic instance.
        topics: A selection of topics to visualize
        top_n_topics: Only select the top n most frequent topics
        custom_labels: If bool, whether to use custom topic labels that were defined using 
                       `topic_model.set_topic_labels`.
                       If `str`, it uses labels from other aspects, e.g., "Aspect1".
        title: Title of the plot.
        width: The width of the figure.
        height: The height of the figure.

    Examples:

    To visualize the topics simply run:

    ```python
    topic_model.visualize_topics()
    ```

    Or if you want to save the resulting figure:

    ```python
    fig = topic_model.visualize_topics()
    fig.write_html("path/to/file.html")
    ```
    <iframe src="../../getting_started/visualization/viz.html"
    style="width:1000px; height: 680px; border: 0px;""></iframe>
    """
    # Select topics based on top_n and topics args
    freq_df = topic_model.get_topic_freq()
    freq_df = freq_df.loc[freq_df.Topic != -1, :]
    if topics is not None:
        topics = list(topics)
    elif top_n_topics is not None:
        topics = sorted(freq_df.Topic.to_list()[:top_n_topics])
    else:
        topics = sorted(freq_df.Topic.to_list())

    # Extract topic words and their frequencies
    topic_list = sorted(topics)
    frequencies = [topic_model.topic_sizes_[topic] for topic in topic_list]
    if isinstance(custom_labels, str):
        words = [[[str(topic), None]] + topic_model.topic_aspects_[custom_labels][topic] for topic in topic_list]
        words = ["_".join([label[0] for label in labels[:4]]) for labels in words]
        words = [label if len(label) < 30 else label[:27] + "..." for label in words]
    elif custom_labels and topic_model.custom_labels_ is not None:
        words = [topic_model.custom_labels_[topic + topic_model._outliers] for topic in topic_list]
    else:
        words = [" | ".join([word[0] for word in topic_model.get_topic(topic)[:5]]) for topic in topic_list]

    # Embed c-TF-IDF into 2D
    all_topics = sorted(list(topic_model.get_topics().keys()))
    indices = np.array([all_topics.index(topic) for topic in topics])

    if topic_model.topic_embeddings_ is not None:
        embeddings = topic_model.topic_embeddings_[indices]
        embeddings = UMAP(n_neighbors=2, n_components=2, metric='cosine', random_state=42).fit_transform(embeddings)
    else:
        embeddings = topic_model.c_tf_idf_.toarray()[indices]
        embeddings = MinMaxScaler().fit_transform(embeddings)
        embeddings = UMAP(n_neighbors=2, n_components=2, metric='hellinger', random_state=42).fit_transform(embeddings)

    # Visualize with plotly
    df = pd.DataFrame({"x": embeddings[:, 0], "y": embeddings[:, 1],
                       "Topic": topic_list, "Words": words, "Size": frequencies})
    return _plotly_topic_visualization(df, topic_list, title, width, height)
def _plotly_topic_visualization(df: pd.DataFrame,
                                topic_list: List[str],
                                title: str,
                                width: int,
                                height: int):
    """ Create plotly-based visualization of topics with a slider for topic selection """

    def get_color(topic_selected):
        if topic_selected == -1:
            marker_color = ["#B0BEC5" for _ in topic_list]
        else:
            marker_color = ["red" if topic == topic_selected else "#B0BEC5" for topic in topic_list]
        return [{'marker.color': [marker_color]}]

    # Prepare figure range
    x_range = (df.x.min() - abs((df.x.min()) * .15), df.x.max() + abs((df.x.max()) * .15))
    y_range = (df.y.min() - abs((df.y.min()) * .15), df.y.max() + abs((df.y.max()) * .15))

    # Plot topics
    fig = px.scatter(df, x="x", y="y", size="Size", size_max=40, template="simple_white", labels={"x": "", "y": ""},
                     hover_data={"Topic": True, "Words": True, "Size": True, "x": False, "y": False})
    fig.update_traces(marker=dict(color="#B0BEC5", line=dict(width=2, color='DarkSlateGrey')))

    # Update hover order
    fig.update_traces(hovertemplate="<br>".join(["<b>Topic %{customdata[0]}</b>",
                                                 "%{customdata[1]}",
                                                 "Size: %{customdata[2]}"]))

    # Create a slider for topic selection
    steps = [dict(label=f"Topic {topic}", method="update", args=get_color(topic)) for topic in topic_list]
    sliders = [dict(active=0, pad={"t": 50}, steps=steps)]

    # Stylize layout
    fig.update_layout(
        title={
            'text': f"{title}",
            'y': .95,
            'x': 0.5,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': dict(
                size=22,
                color="Black")
        },
        width=width,
        height=height,
        hoverlabel=dict(
            bgcolor="white",
            font_size=16,
            font_family="Rockwell"
        ),
        xaxis={"visible": False},
        yaxis={"visible": False},
        sliders=sliders
    )

    # Update axes ranges
    fig.update_xaxes(range=x_range)
    fig.update_yaxes(range=y_range)

    # Add grid in a 'plus' shape
    fig.add_shape(type="line",
                  x0=sum(x_range) / 2, y0=y_range[0], x1=sum(x_range) / 2, y1=y_range[1],
                  line=dict(color="#CFD8DC", width=2))
    fig.add_shape(type="line",
                  x0=x_range[0], y0=sum(y_range) / 2, x1=x_range[1], y1=sum(y_range) / 2,
                  line=dict(color="#9E9E9E", width=2))
    fig.add_annotation(x=x_range[0], y=sum(y_range) / 2, text="D1", showarrow=False, yshift=10)
    fig.add_annotation(y=y_range[1], x=sum(x_range) / 2, text="D2", showarrow=False, xshift=10)
    fig.data = fig.data[::-1]

    return fig

In [295]:
# fitting the models
model_jury = fit_model(df_jury)
model_csop = fit_model(df_csop)


1. If you print out model.get_topic(-1), you will get all documents that did not have any topics assigned.

2. The great thing about HDBSCAN (which is used for clustering) is that not all documents are forced towards a certain cluster. 
If no cluster could be found, then it is simply an outlier.

3. The topics have multiple dimensions, but they are reduced to two dimensions for the purpose of plotting the graph 
(I AM NOT SURE HOW TO FACTOR THIS WHILE READING THE GRAPH)

Further Steps:

There is a function called topics_over_time(self,docs: List[str],timestamps: Union[List[str],List[int]],nr_bins: int = None,datetime_format:str= None,evolution_tuning: bool = True,global_tuning: bool = True)

(link https://github.com/MaartenGr/BERTopic/blob/master/bertopic/_bertopic.py#L670)

within BERTopic, which takes returns how the topics vary by timestamp. Since we have many timestamps, I am thinking of implementing the following approach:

1. Find the conversation duration: Last Timestamp - First Timestamp. Divide this by 3, as we are assuming a conversation should have around 3 parts.

2. Move the boundaries of the three topics until the top topics contain x% of different words.


Issues (PRIYA NEEDS HELP!)

1. Does the above idea seem logical?
2. The topics_over_time function is breaking when I am trying to Union(2nd parameter). I've tried to figure out why but to no avail (yet) - in the code commented below :(. Some help here would be great!


In [305]:
# #convert time stamps into equal time intervals

# # Assuming your DataFrame is called 'df' and the timestamp column is 'timestamp'
# # Convert timestamp column to datetime if needed
# df_jury['timestamp'] = pd.to_datetime(df_jury['timestamp'])

# # Calculate the time difference between the first and last timestamp
# time_diff = df_jury['timestamp'].iloc[-1] - df_jury['timestamp'].iloc[0]

# # Determine the duration for each interval
# interval_duration = time_diff / 3

# # Create three intervals by slicing the DataFrame
# interval1 = df_jury[df_jury['timestamp'] <= df_jury['timestamp'].iloc[0] + interval_duration]
# interval2 = df_jury[(df_jury['timestamp'] > df_jury['timestamp'].iloc[0] + interval_duration) &
#                (df_jury['timestamp'] <= df_jury['timestamp'].iloc[0] + 2 * interval_duration)]
# interval3 = df_jury[df_jury['timestamp'] > df_jury['timestamp'].iloc[0] + 2 * interval_duration]


# #Topics over time
# union_data: Union[str, int] = interval1['timestamp'].tolist() + interval1['message'].tolist() 
# model_jury.topics_over_time(interval1['message'].tolist(),union_data)

In [296]:
#For CSOP
get_top_topics(model_csop,3)
visualize_topics(model_csop, None, None,False,"<b>Intertopic Distance Map - CSOP</b>",650,650)

Topic Number 0
nan
wee
thas
sho
fa
corret




-------------------
Topic Number 1
moving
move
them
moves
person
moved
mover
dont
around
stop
-------------------
Topic Number 2
score
high
our
scores
down
the
higher
went
lower
highest
-------------------


In [298]:
#For Juries
get_top_topics(model_jury,3)
visualize_topics(model_jury, None, None,False,"<b>Intertopic Distance Map - Jury</b>",650,650)

Topic Number 0
german
germany
speak
speaking
english
american
mom
can
mother
when
-------------------
Topic Number 1
mom
mother
moms
mothers
included
house
conversation
include
conversations
hosting
-------------------
Topic Number 2
him
she
help
time
her
busy
asking
he
it
on
-------------------
